In [2]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder


In [4]:

ROOT_DIR = r'C:\Users\HP\Documents\Assessment'
sys.path.append(ROOT_DIR)
from utils.ingestion import load_csv

df = load_csv(os.path.join(ROOT_DIR, 'data', 'raw', 'data_80.csv'))
df.head()

✅ Loaded CSV: 2400 rows × 24 columns


,Date,Stock Index,Open Price,Close Price,Daily High,Daily Low,Trading Volume,GDP Growth (%),Inflation Rate (%),Unemployment Rate (%),...,Forex USD/EUR,Forex USD/JPY,Crude Oil Price (USD per Barrel),Gold Price (USD per Ounce),Real Estate Index,Retail Sales (Billion USD),Bankruptcy Rate (%),Mergers & Acquisitions Deals,Venture Capital Funding (Billion USD),Consumer Spending (Billion USD)
0,2004-12-06,NASDAQ,2185.43,2141.14,2224.67,2113.20,113169924,2.96,7.94,13.10,...,0.84,124.38,43.44,803.27,170.95,4154,5.63,31,99.39,13920
1,2003-04-05,S&P 500,4157.17,4121.73,4197.88,4100.04,8875728,9.51,7.71,9.04,...,1.06,106.34,102.30,1394.83,326.65,4051,4.97,27,82.40,8993
2,2004-12-22,Dow Jones,2436.83,2410.39,2466.63,2370.78,182890777,4.00,4.69,11.86,...,1.07,136.18,57.19,2355.57,409.71,6463,2.93,45,71.56,3556
3,2000-09-08,Dow Jones,1375.43,1364.04,1421.50,1336.35,39372961,7.00,2.36,8.62,...,1.16,133.60,55.96,1811.67,360.46,5043,7.47,10,14.24,13709
4,2006-11-10,Dow Jones,2984.96,2990.64,3030.46,2978.91,978906610,8.08,4.11,5.94,...,1.44,107.19,134.84,2351.80,123.47,7739,4.79,24,33.16,5747


In [3]:
# Numeric — fill with median
numeric_cols = df.select_dtypes(include='number').columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical — fill with mode
categorical_cols = df.select_dtypes(exclude='number').columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"✅ Nulls handled")
print(f"   Nulls remaining: {df.isnull().sum().sum()}")

✅ Nulls handled
   Nulls remaining: 0


In [4]:
# Numeric — fill with median
numeric_cols = df.select_dtypes(include='number').columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical — fill with mode
categorical_cols = df.select_dtypes(exclude='number').columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Nulls remaining: {df.isnull().sum().sum()}")

Nulls remaining: 0


In [5]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicates")
print(f"Rows remaining: {len(df)}")

Removed 0 duplicates
Rows remaining: 2400


In [6]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f"Date range: {df['Date'].min()} → {df['Date'].max()}")

Date range: 2000-01-01 00:00:00 → 2008-03-18 00:00:00


In [7]:
#standardize
df['Stock Index'] = df['Stock Index'].astype(str).str.strip().str.upper()
print(f"Stock Index values: {df['Stock Index'].unique().tolist()}")

Stock Index values: ['DOW JONES', 'S&P 500', 'NASDAQ']


In [8]:
for col in numeric_cols:
    Q1    = df[col].quantile(0.25)
    Q3    = df[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

print("✅ Outliers capped using IQR method")

✅ Outliers capped using IQR method


In [15]:
# Ensure Date is datetime first
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Quarter']    = df['Date'].dt.quarter
df['Month_Name'] = df['Date'].dt.strftime('%b')

print("✅ Date features extracted")
print(f"   + Year, Month, Quarter, Month_Name")
print(f"   Current columns: {df.shape[1]}")

✅ Date features extracted
   + Year, Month, Quarter, Month_Name
   Current columns: 34


In [10]:
# Feature 1 — Daily Return
df['Daily_Return (%)'] = ((df['Close Price'] - df['Open Price']) / df['Open Price']) * 100

# Feature 2 — Misery Index
df['Misery_Index'] = df['Inflation Rate (%)'] + df['Unemployment Rate (%)']

# Feature 3 — Real Interest Rate
df['Real_Interest_Rate (%)'] = df['Interest Rate (%)'] - df['Inflation Rate (%)']

# Feature 4 — Volatility Band
df['Volatility_Band (%)'] = ((df['Daily High'] - df['Daily Low']) / df['Close Price']) * 100

# Feature 5 — Economic Health Score
df['Economic_Health_Score'] = (
    df['GDP Growth (%)'] * 0.4 +
    df['Consumer Confidence Index'] * 0.3 +
    (df['Corporate Profits (Billion USD)'] / df['Corporate Profits (Billion USD)'].max()) * 100 * 0.3
).round(4)

print("✅ Engineered features added:")
print("   + Daily_Return (%)")
print("   + Misery_Index")
print("   + Real_Interest_Rate (%)")
print("   + Volatility_Band (%)")
print("   + Economic_Health_Score")

✅ Engineered features added:
   + Daily_Return (%)
   + Misery_Index
   + Real_Interest_Rate (%)
   + Volatility_Band (%)
   + Economic_Health_Score


In [11]:
#encode categoricals
le = LabelEncoder()
df['Stock_Index_encoded'] = le.fit_transform(df['Stock Index'].astype(str))
print(f"✅ Stock Index encoded")
print(f"   Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

✅ Stock Index encoded
   Mapping: {'DOW JONES': 0, 'NASDAQ': 1, 'S&P 500': 2}


In [12]:
print("=" * 50)
print("   CLEANING & TRANSFORMATION COMPLETE")
print("=" * 50)
print(f"   Rows    : {df.shape[0]}")
print(f"   Columns : {df.shape[1]}")
print(f"   Nulls   : {df.isnull().sum().sum()}")
print(f"   Dupes   : {df.duplicated().sum()}")
print("=" * 50)
df.head()

   CLEANING & TRANSFORMATION COMPLETE
   Rows    : 2400
   Columns : 34
   Nulls   : 0
   Dupes   : 0


,Date,Stock Index,Open Price,Close Price,Daily High,Daily Low,Trading Volume,GDP Growth (%),Inflation Rate (%),Unemployment Rate (%),...,Year,Month,Quarter,Month_Name,Daily_Return (%),Misery_Index,Real_Interest_Rate (%),Volatility_Band (%),Economic_Health_Score,Stock_Index_encoded
0,2000-01-01,DOW JONES,2128.75,2138.48,2143.70,2100.55,2670411,-0.37,6.06,6.10,...,2000,1,1,Jan,0.457076,12.16,0.00,2.017788,43.9240,0
1,2000-01-03,DOW JONES,1987.92,1985.26,2022.28,1978.37,315284661,5.54,9.13,2.60,...,2000,1,1,Jan,-0.133808,11.73,-8.31,2.211801,57.1935,0
2,2000-01-04,DOW JONES,4625.02,4660.47,4665.26,4595.46,13098297,10.00,3.77,2.20,...,2000,1,1,Jan,0.766483,5.97,-0.06,1.497703,62.7030,0
3,2000-01-06,S&P 500,2087.80,2124.76,2153.18,2085.18,82664194,1.42,6.08,3.24,...,2000,1,1,Jan,1.770285,9.32,-4.99,3.200361,42.1210,2
4,2000-01-07,DOW JONES,4037.59,3996.40,4055.78,3948.97,653722138,7.64,6.24,4.52,...,2000,1,1,Jan,-1.020163,10.76,2.00,2.672655,50.2680,0


In [16]:
# os.makedirs('data/processed', exist_ok=True)
# df.to_parquet('data/processed/processed.parquet', index=False, engine='fastparquet')
# # Verify
# size = os.path.getsize('data/processed/processed.parquet') / 1024
# print(f"✅ File saved: data/processed/processed.parquet")

# print(f"   Rows    : {df.shape[0]}")
# print(f"   Columns : {df.shape[1]}")
# print(f"   Size    : {size:.1f} KB")

save_path = os.path.join(ROOT_DIR, 'data', 'processed', 'processed.parquet')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
df.to_parquet(save_path, index=False, engine='pyarrow')

size = os.path.getsize(save_path) / 1024
print(f"✅ File saved: {save_path}")
print(f"   Rows    : {df.shape[0]}")
print(f"   Columns : {df.shape[1]}")
print(f"   Size    : {size:.1f} KB")

✅ File saved: C:\Users\HP\Documents\assessment\data\processed\processed.parquet
   Rows    : 2400
   Columns : 34
   Size    : 371.7 KB


In [14]:
df_check = pd.read_parquet('data/processed/processed.parquet')
print("✅ Parquet verified:")
print(f"   Shape: {df_check.shape}")
df_check.head()

✅ Parquet verified:
   Shape: (2400, 34)


,Date,Stock Index,Open Price,Close Price,Daily High,Daily Low,Trading Volume,GDP Growth (%),Inflation Rate (%),Unemployment Rate (%),...,Year,Month,Quarter,Month_Name,Daily_Return (%),Misery_Index,Real_Interest_Rate (%),Volatility_Band (%),Economic_Health_Score,Stock_Index_encoded
0,2000-01-01,DOW JONES,2128.75,2138.48,2143.70,2100.55,2670411,-0.37,6.06,6.10,...,2000,1,1,Jan,0.457076,12.16,0.00,2.017788,43.9240,0
1,2000-01-03,DOW JONES,1987.92,1985.26,2022.28,1978.37,315284661,5.54,9.13,2.60,...,2000,1,1,Jan,-0.133808,11.73,-8.31,2.211801,57.1935,0
2,2000-01-04,DOW JONES,4625.02,4660.47,4665.26,4595.46,13098297,10.00,3.77,2.20,...,2000,1,1,Jan,0.766483,5.97,-0.06,1.497703,62.7030,0
3,2000-01-06,S&P 500,2087.80,2124.76,2153.18,2085.18,82664194,1.42,6.08,3.24,...,2000,1,1,Jan,1.770285,9.32,-4.99,3.200361,42.1210,2
4,2000-01-07,DOW JONES,4037.59,3996.40,4055.78,3948.97,653722138,7.64,6.24,4.52,...,2000,1,1,Jan,-1.020163,10.76,2.00,2.672655,50.2680,0
